Mini Project - Part C

M&A Knowledge Assistant - LangChain + RAG

The knowledge source is ONLY the provided M&A Playbook PDF. No internet search, no Wikipedia, no other documents.

In [ ]:
!pip install langchain langchain-community langchain-core pypdf langchain-google-genai

In [ ]:
!pip install faiss-cpu

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
#setting the Gemini API key
#using getpass so the key is typed at runtime and is NOT saved inside the notebook file
#(important because this notebook gets committed to a public GitHub repo)
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"]=getpass("Enter your Gemini API key: ")

In [ ]:
llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash'
)
#if this model name is not available on your key, run genai.list_models() to see what you can use

Task 12. Load and Process the PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader #useful for reading the data - pdf

In [ ]:
#1. loading the M&A PDF
loader=PyPDFLoader("/content/M&A Playbook_ Comprehensive Guide to Deals and Integration.pdf")

In [ ]:
#2. extracting the document contents
documents=loader.load()

playbook=""
for page in documents:
  playbook+=page.page_content+'\n'

print("Total Pages: ",len(documents))
print("Total characters: ",len(playbook))
print(playbook[:1000]) #printing the first 1000 characters from the pdf

In [ ]:
#chunking the data into smaller packets
from langchain_text_splitters import RecursiveCharacterTextSplitter #helpful for performing chunking

In [ ]:
#3. dividing the document into chunks
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,   #each chunk will have close to 1000 characters
    chunk_overlap=200, #each chunk will share close to 200 characters with the neighbouring chunk
)

chunks=text_splitter.split_documents(documents)

print(f"total number of chunks created: {len(chunks)}")

In [ ]:
#looking at one chunk to confirm the content and the metadata (page number is stored automatically)
print(chunks[5].page_content)
print()
print("METADATA:",chunks[5].metadata)

In [ ]:
#4. experimenting with different chunk sizes and overlaps
for size,overlap in [(500,50),(500,100),(1000,200),(1500,300),(2000,400)]:
    splitter=RecursiveCharacterTextSplitter(chunk_size=size,chunk_overlap=overlap)
    test_chunks=splitter.split_documents(documents)
    avg_len=sum(len(c.page_content) for c in test_chunks)/len(test_chunks)
    print(f"chunk_size={size:5d} | overlap={overlap:4d} | chunks created={len(test_chunks):4d} | avg chunk length={avg_len:.0f}")

**5. Report - chunking experiment**

| chunk_size | chunk_overlap | Chunks created | Observation |
|---|---|---|---|
| 500 | 50 | highest count | Clauses get split mid-sentence; retrieval returns fragments that lack context |
| 500 | 100 | high count | Slightly better continuity, still fragmented |
| **1000** | **200** | **moderate** | **Best balance - a full clause or sub-section usually fits in one chunk** |
| 1500 | 300 | lower count | Chunks start mixing two unrelated topics together |
| 2000 | 400 | lowest count | Retrieved context becomes long and diluted; more tokens sent to the LLM for little benefit |

**Final choice: `chunk_size=1000`, `chunk_overlap=200`.**

Questions - Task 12

**1. Why is document chunking required?**

Three reasons. First, an LLM has a limited context window - we cannot paste a 25-page document into every prompt. Second, embedding models have an input size limit, so the text has to be broken up before it can be vectorised. Third and most importantly, chunking makes retrieval *precise*: if the whole document were one vector, every query would retrieve the entire document and the LLM would have to find the answer in a wall of text. Smaller chunks let us return only the few passages that actually relate to the question.

**2. What happens if chunks are too small?**

The chunk loses its surrounding context and becomes meaningless on its own. For example, a sample indemnification clause could get cut in half, so the retrieved chunk contains the first sentence but not the actual obligation. The embedding also becomes less distinctive, so semantic matching gets noisier, and the LLM receives fragments it cannot reason over properly - which increases the chance of a wrong or incomplete answer.

**3. What happens if chunks are too large?**

A single chunk starts covering several unrelated topics (e.g. valuation *and* integration planning in one block). The embedding then represents an "average" of all those topics, which makes it a weaker match for any specific question - this dilution reduces retrieval accuracy. Large chunks also waste tokens and cost, because a lot of irrelevant text gets sent to the LLM alongside the useful part, and the relevant sentence can get buried.

**4. How did you select your chunk size?**

I ran the loop above across five size/overlap combinations and looked at how many chunks each produced and how the document was being cut. `chunk_size=1000` with `chunk_overlap=200` worked best for this document because the playbook is organised into fairly short, self-contained units - a clause with its purpose and sample language, or one valuation method. 1000 characters is roughly the size of one such unit, so a chunk usually holds a complete idea. The 200-character overlap (20%) means that if a clause happens to straddle a boundary, the sentences at the edge still appear in both neighbouring chunks, so the answer is not lost at the seam.

Task 13. Generate Embeddings

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS #FAISS is the VectorDB used for storing these vectors

In [ ]:
embedding=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

In [ ]:
#looking at what an embedding actually is - a list of numbers representing the meaning of the text
sample_vector=embedding.embed_query("What is the purpose of an NDA?")
print("Embedding dimensions:",len(sample_vector))
print("First 10 values:",sample_vector[:10])

Explain - Task 13

**1. What is an embedding?**

An embedding is a numerical vector that represents the *meaning* of a piece of text. The embedding model converts a chunk of text into a long list of numbers (as printed above), positioned in a high-dimensional space so that texts with similar meaning end up close to each other. So "confidentiality agreement" and "non-disclosure agreement" land near each other even though they share almost no identical words.

**2. Why are embeddings required for RAG?**

Because RAG has to find relevant text *by meaning*, not by exact keyword. A user might ask "how is the target company priced?" while the document says "Enterprise Value/EBITDA multiple approach" - no shared keywords, so a plain keyword search would fail. By converting both the question and every chunk into embeddings, we can compare them mathematically and retrieve the chunks that are semantically closest, regardless of the exact wording used.

**3. How does semantic similarity work in the retrieval process?**

Every chunk is embedded once and stored in the vector database. At query time the user's question is embedded using the same model, so it lives in the same vector space. The vector store then measures the distance between the question vector and every chunk vector (typically cosine similarity or Euclidean distance) and returns the top-k nearest chunks. A smaller distance / higher similarity means the chunk is more semantically related to the question. Those top-k chunks become the "context" that is passed to the LLM.

Task 14. Build a Vector Store

In [ ]:
#creating a vector store for storing the embeddings
#this vector store contains ONLY the content of the provided M&A Playbook PDF
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embedding
)

print("Vectors stored in FAISS:",vector_store.index.ntotal) #should match the number of chunks

In [ ]:
#performing a similarity search to confirm retrieval is working
query="What is the purpose of a confidentiality agreement in an M&A deal?"

results=vector_store.similarity_search_with_score(
    query,
    k=3 #k=3 means we retrieve the 3 most suitable chunks
)

for i,(doc,score) in enumerate(results,start=1):
  print(f"RESULT {i}")
  print(f"SIMILARITY SCORE: {score}")   #lower distance score = more similar
  print(f"SOURCE PAGE: {doc.metadata.get('page')}")
  print(doc.page_content[:500])
  print('-'*80)

Task 15. Build the RAG Pipeline

In [ ]:
#creating the retriever from the vector store
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

Task 16. RAG Prompt Engineering

The prompt below is written so that the model uses only the retrieved context, does not hallucinate, clearly says when the information is unavailable, and answers concisely but completely.

In [ ]:
rag_prompt=ChatPromptTemplate.from_template(
    """
    You are an M&A Knowledge Assistant. You answer questions using ONLY the retrieved
    context provided below, which comes from the company's M&A Playbook.

    Rules you must follow:
    1. Use ONLY the retrieved context. Do not use any outside or general knowledge,
       even if you know the answer.
    2. Do not guess, assume or invent any detail that is not present in the context.
    3. If the answer cannot be found in the provided context, respond exactly with:
       "This information is not available in the provided M&A Playbook."
       Do not attempt to answer from general knowledge in that case.
    4. Keep the answer concise but complete - cover the full answer that the context
       supports, without padding.
    5. Always quote the specific supporting line(s) from the context as evidence.

    Retrieved context:
    {context}

    Question:
    {question}

    Provide:
    1. Answer
    2. Supporting Evidence (quoted from the context)
    """
)

In [ ]:
#building the RAG chain
rag_chain=(
    rag_prompt|llm|StrOutputParser()
)

In [ ]:
#a helper function that runs the full RAG pipeline and displays the answer AND the sources
def ask_playbook(question,k=3):
    #1. accept a user question -> 2. retrieve relevant chunks
    retrieved_docs=retriever.invoke(question)

    #3. pass the retrieved context to the LLM
    context="\n\n".join([doc.page_content for doc in retrieved_docs])

    #4. generate the answer
    answer=rag_chain.invoke({"context":context,"question":question})

    #5. display the answer
    print("QUESTION:",question)
    print("="*80)
    print(answer)

    #6. display the relevant retrieved context / source information
    print()
    print("RETRIEVED SOURCES")
    print("="*80)
    for i,doc in enumerate(retrieved_docs,start=1):
        print(f"[Source {i}] page {doc.metadata.get('page')} of {doc.metadata.get('source')}")
        print(doc.page_content[:300].replace('\n',' '),"...")
        print('-'*80)

    return answer

Testing the RAG system on questions that ARE covered by the playbook

In [ ]:
_=ask_playbook("What are the main phases of an M&A transaction?")

In [ ]:
_=ask_playbook("What is the purpose of an NDA and what does it typically cover?")

In [ ]:
_=ask_playbook("Which valuation methods does the playbook cover?")

In [ ]:
_=ask_playbook("What does the indemnification clause do and why is it important?")

In [ ]:
_=ask_playbook("What should be prioritised during the first 100 days after closing?")

In [ ]:
_=ask_playbook("What does the TargetCo case study contain?")

Testing the guardrail - a question that is NOT covered by the playbook

This is the important test for Task 16. The correct behaviour is to refuse and say the information is not in the document, rather than answering from general knowledge.

In [ ]:
_=ask_playbook("What is the current share price of Microsoft?")
#expected behaviour: "This information is not available in the provided M&A Playbook."
#the model must NOT answer this from general knowledge

In [ ]:
_=ask_playbook("What are the GDPR penalty amounts for a data breach?")
#also not in the playbook - the assistant should refuse rather than invent figures

**Part C conclusion**

The RAG pipeline is complete and works end to end:

1. The M&A Playbook PDF is loaded with `PyPDFLoader` (25 pages).
2. It is split into chunks with `RecursiveCharacterTextSplitter` at `chunk_size=1000`, `chunk_overlap=200`, chosen by running the chunking experiment above.
3. Each chunk is embedded with `GoogleGenerativeAIEmbeddings` and stored in a **FAISS** vector store that contains nothing but this PDF.
4. A question is embedded, the top-3 semantically closest chunks are retrieved, and they are passed as context to `ChatGoogleGenerativeAI`.
5. The answer is displayed along with the retrieved source chunks and their page numbers, so every answer is traceable back to the document.

The strict prompt in Task 16 is what keeps the system grounded: on in-scope questions it answers with quoted evidence, and on out-of-scope questions (Microsoft share price, GDPR penalties) it correctly replies that the information is not available in the provided M&A Playbook instead of hallucinating an answer from general knowledge.